In [2]:
import pandas as pd
import requests
import os
import re
import json

from collections import defaultdict
from dotenv import load_dotenv

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

<br> <br> <br>

### Read the the .csv file

In [10]:
lag_features_path = "../../data/azure_pm/lag_features/machine_98_lag_features.csv"
df_machine_98 = pd.read_csv(lag_features_path)
print(f"Loaded shape: {df_machine_98.shape}")
df_machine_98.head()

Loaded shape: (8757, 65)


,datetime,volt,rotate,pressure,vibration,errorID,comp,failure,target,volt_lag_1h,...,error_count_6h,error_count_24h,maint_count_6h,maint_count_24h,hour,day_of_week,is_weekend,is_working_hours,hours_since_maint,hours_since_error
0,2015-01-01 06:00:00,0.303137,0.604476,0.271993,0.621000,0,0,0,0,0.000000,...,0.0,0.0,0.0,0.0,6,3,0,0,0,0
1,2015-01-01 07:00:00,0.455312,0.635538,0.480756,0.645421,0,0,0,0,0.303137,...,0.0,0.0,0.0,0.0,7,3,0,0,1,1
2,2015-01-01 08:00:00,0.449470,0.540973,0.427196,0.356126,0,0,0,0,0.455312,...,0.0,0.0,0.0,0.0,8,3,0,1,2,2
3,2015-01-01 09:00:00,0.408248,0.687752,0.378658,0.254123,0,0,0,0,0.449470,...,0.0,0.0,0.0,0.0,9,3,0,1,3,3
4,2015-01-01 10:00:00,0.486041,0.518746,0.454206,0.351050,0,0,0,0,0.408248,...,0.0,0.0,0.0,0.0,10,3,0,1,4,4


<br> <br> <br>

## Generate Taxonomy

In [6]:
def generate_taxonomy_with_groq(columns_list, api_key, model="llama3-70b-8192", output_file="groq_taxonomy.json"):
    output_file = f"{output_file}"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

    # API endpoint
    endpoint = "https://api.groq.com/openai/v1/chat/completions"

    # Create the prompt for the taxonomy
    prompt = f"""
    **Task**:  
    Create a hierarchical taxonomy for dataset features.

    **Instructions**:
    
    1. **Organize Features Hierarchically**:  
    - Group dataset features based on their semantic meaning and relationships.
    - The taxonomy should show clear parent-child relationships between features.

    2. **Output Must Be Valid JSON**:
    - Ensure the response is in **JSON format**.
    - No extra text or explanations.

    **Example Format**:
    {{
        "Type": [1, "Root"],
        "Temperature": [1, "Root"],
        "Air_temperature": [2, "Temperature"]
        "Air_temperature_combination": [3, "Air_temperature"]
        
    }}

    **Input Features**:
    {', '.join(columns_list)}
    """

    # Create the request payload
    payload = {
        "model": model,  # Using the model specified as parameter
        "messages": [
            {
                "role": "system",
                "content": "You are a data scientist who organizes features into taxonomies.",
            },
            {"role": "user", "content": prompt},
        ],
        "temperature": 0.2,
    }

    # Make the API call
    response = requests.post(endpoint, headers=headers, json=payload)

    if response.status_code == 200:
        result = response.json()
        content = result["choices"][0]["message"]["content"]

        # Extract JSON from the response
        json_match = re.search(r"{.*}", content, re.DOTALL)
        if json_match:
            taxonomy_json_str = json_match.group(0)
            taxonomy = json.loads(taxonomy_json_str)
        else:
            # If no JSON pattern found, try to parse the whole content
            try:
                taxonomy = json.loads(content)
            except:
                raise ValueError("Failed to parse JSON from API response")

        # Save to file
        with open(output_file, "w") as f:
            json.dump(taxonomy, f, indent=2)

        return taxonomy
    else:
        error_message = f"API request failed with status code {response.status_code}"
        try:
            error_details = response.json()
            error_message += f": {error_details}"
        except:
            pass
        raise Exception(error_message)


In [7]:
def generate_taxonomy_with_openai(filename, features, openai_api_key):
    prompt = f"""
    **Task**:  
    Create a hierarchical taxonomy for dataset features.

    **Instructions**:
    
    1. **Organize Features Hierarchically**:  
    - Group dataset features based on their semantic meaning and relationships.
    - The taxonomy should show clear parent-child relationships between features.

    2. **Output Must Be Valid JSON**:
    - Ensure the response is in **JSON format**.
    - No extra text or explanations.

    3. The hierarchy can be at any levels (e.g. 2,3,4,etc) but should be logical and meaningful.

    **Example Format**:
    {{
        "Type": [1, "Root"],
        "Temperature": [1, "Root"],
        "Air_temperature": [2, "Temperature"]

    }}

    **Input Features**:
    {json.dumps(features, indent=2)}
    """

    openai_url = "https://api.openai.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {openai_api_key}",
        "Content-Type": "application/json",
    }
    data = {
        "model": "gpt-4o",
        "messages": [
            {
                "role": "system",
                "content": "Respond only with valid JSON, no extra text.",
            },
            {"role": "user", "content": prompt},
        ],
        "response_format": {
            "type": "json_object"
        },  # ✅ Changed from "json" to "json_object"
        "temperature": 0,
    }

    response = requests.post(openai_url, headers=headers, json=data)

    if response.status_code == 200:
        raw_content = response.json()["choices"][0]["message"]["content"].strip()

        # print("🔍 Raw OpenAI Response:\n", raw_content)  # Debugging step

        # Extract only JSON if extra text is present
        json_match = re.search(r"\{.*\}", raw_content, re.DOTALL)
        if json_match:
            raw_content = json_match.group(0)  # Extract JSON only

        # Try to parse as JSON
        try:
            taxonomy = json.loads(raw_content)
        except json.JSONDecodeError:
            raise RuntimeError(
                f"OpenAI API returned an invalid JSON response:\n{raw_content}"
            )

        # Save taxonomy to a file
        with open(f"{filename}", "w") as f:
            json.dump(taxonomy, f, indent=2)

        return taxonomy
    else:
        raise RuntimeError(
            f"Error from OpenAI API: {response.status_code} {response.text}"
        )


In [15]:
def generate_taxonomy_with_openai_edited(filename, features, root_features, openai_api_key):
    prompt = f"""
    **Task**:  
    Create a hierarchical taxonomy for dataset features.

    **Instructions**:
    
    1. **Organize Features Hierarchically**:  
    - Group dataset features based on their semantic meaning and relationships.
    - The taxonomy should show clear parent-child relationships between features.
    - The hierachy MUST NOT include features that are not existed in the dataset's features  
      Dataset's list of feature: {features} 
    -The roots of the hierarchy are {root_features}. These features can not have parents.


    2. **Output Must Be Valid JSON**:
    - Ensure the response is in **JSON format**.
    - No extra text or explanations.

    3. The hierarchy can be at any levels (e.g. 2,3,4,etc) but should be logical and meaningful.

    **Example Format**:
    {{
        "Type": [1, "Root"],
        "Temperature": [1, "Root"],
        "Air_temperature": [2, "Temperature"]

    }}

    **Input Features**:
    {json.dumps(features, indent=2)}
    """

    openai_url = "https://api.openai.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {openai_api_key}",
        "Content-Type": "application/json",
    }
    data = {
        "model": "gpt-4o",
        "messages": [
            {
                "role": "system",
                "content": "Respond only with valid JSON, no extra text.",
            },
            {"role": "user", "content": prompt},
        ],
        "response_format": {
            "type": "json_object"
        },  # ✅ Changed from "json" to "json_object"
        "temperature": 0,
    }

    response = requests.post(openai_url, headers=headers, json=data)

    if response.status_code == 200:
        raw_content = response.json()["choices"][0]["message"]["content"].strip()

        # print("🔍 Raw OpenAI Response:\n", raw_content)  # Debugging step

        # Extract only JSON if extra text is present
        json_match = re.search(r"\{.*\}", raw_content, re.DOTALL)
        if json_match:
            raw_content = json_match.group(0)  # Extract JSON only

        # Try to parse as JSON
        try:
            taxonomy = json.loads(raw_content)
        except json.JSONDecodeError:
            raise RuntimeError(
                f"OpenAI API returned an invalid JSON response:\n{raw_content}"
            )

        # Save taxonomy to a file
        with open(f"{filename}", "w") as f:
            json.dump(taxonomy, f, indent=2)

        return taxonomy
    else:
        raise RuntimeError(
            f"Error from OpenAI API: {response.status_code} {response.text}"
        )


root_features = ["datetime", "volt", "rotate", "pressure", "vibration", "errorID", "comp", "failure"]
gpt4o_edited_taxonomy = generate_taxonomy_with_openai_edited(filename="../../taxonomy/gpt4o_edited.json", 
                                                             features=df_machine_98.columns.tolist(), 
                                                             root_features = root_features,
                                                             openai_api_key=openai_api_key)


In [16]:
print(format_taxonomy(gpt4o_edited_taxonomy))

📌 **Taxonomy Structure:**

├── 📂 datetime
  ├── 📄 hour
  ├── 📄 day_of_week
  ├── 📄 is_weekend
  └── 📄 is_working_hours
├── 📂 volt
  ├── 📄 volt_lag_1h
  ├── 📄 volt_lag_6h
  ├── 📄 volt_lag_12h
  ├── 📄 volt_lag_24h
  ├── 📄 volt_mean_24h
  ├── 📄 volt_std_24h
  ├── 📄 volt_min_24h
  ├── 📄 volt_max_24h
  ├── 📄 volt_mean_6h
  └── 📄 volt_std_6h
├── 📂 rotate
  ├── 📄 rotate_lag_1h
  ├── 📄 rotate_lag_6h
  ├── 📄 rotate_lag_12h
  ├── 📄 rotate_lag_24h
  ├── 📄 rotate_mean_24h
  ├── 📄 rotate_std_24h
  ├── 📄 rotate_min_24h
  ├── 📄 rotate_max_24h
  ├── 📄 rotate_mean_6h
  └── 📄 rotate_std_6h
├── 📂 pressure
  ├── 📄 pressure_lag_1h
  ├── 📄 pressure_lag_6h
  ├── 📄 pressure_lag_12h
  ├── 📄 pressure_lag_24h
  ├── 📄 pressure_mean_24h
  ├── 📄 pressure_std_24h
  ├── 📄 pressure_min_24h
  ├── 📄 pressure_max_24h
  ├── 📄 pressure_mean_6h
  └── 📄 pressure_std_6h
├── 📂 vibration
  ├── 📄 vibration_lag_1h
  ├── 📄 vibration_lag_6h
  ├── 📄 vibration_lag_12h
  ├── 📄 vibration_lag_24h
  ├── 📄 vibration_mean_24h
  ├── 📄 vibra

In [8]:
def format_taxonomy(taxonomy_json):
    """Formats a hierarchical taxonomy into a tree-like string."""
    if isinstance(taxonomy_json, str):
        taxonomy_json = json.loads(taxonomy_json)  # Parse JSON string if needed

    # Build tree structure
    tree = defaultdict(list)
    for feature, (_, parent) in taxonomy_json.items():
        tree[parent].append(feature)

    def display_tree(parent="Root", level=0):
        """Recursively formats the tree structure with icons and connectors."""
        if parent not in tree:
            return ""

        result = ""
        for idx, child in enumerate(tree[parent]):
            is_parent = child in tree  # Check if child has further children
            icon = "📂" if is_parent else "📄"  # Folder for parents, file for leaf nodes
            connector = "└── " if idx == len(tree[parent]) - 1 else "├── "  # Tree connectors
            result += "  " * level + connector + f"{icon} {child}\n"
            result += display_tree(child, level + 1)  # Recursive call for children

        return result

    return f"\n📌 **Taxonomy Structure:**\n\n{display_tree('Root')}".strip()


<br> <br> <br>

### Generate the taxonomy


```python
llama3_taxonomy = generate_taxonomy_with_groq(columns_list=df_machine_98.columns, api_key=groq_api_key, model="llama-3.3-70b-versatile", output_file="../../taxonomy/llama3.json")

gemma2_taxonomy = generate_taxonomy_with_groq(columns_list=df_machine_98.columns, api_key=groq_api_key, model="gemma2-9b-it", output_file="../../taxonomy/gemma2.json")

deepseek_taxonomy = generate_taxonomy_with_groq(columns_list=df_machine_98.columns, api_key=groq_api_key, model="deepseek-r1-distill-llama-70b", output_file="../../taxonomy/deepseek.json")

gpt4o_taxonomy = generate_taxonomy_with_openai(filename="../../taxonomy/gpt4o.json", features=df_machine_98.columns.tolist(), openai_api_key=openai_api_key)```

<br> <br> <br>

## Load the Taxonomy 

In [8]:
with open('../../taxonomy/deepseek.json', 'r') as file:
    deepseek_taxonomy = json.load(file)


with open('../../taxonomy/gemma2.json', 'r') as file:
    gemma2_taxonomy = json.load(file)


with open('../../taxonomy/llama3.json', 'r') as file:
    llama3_taxonomy = json.load(file)


with open('../../taxonomy/gpt4o.json', 'r') as file:
    gpt4o_taxonomy = json.load(file)

In [9]:
deepseek_taxonomy

{'datetime': [1, 'time'],
 'volt': [1, 'volt'],
 'rotate': [1, 'rotate'],
 'pressure': [1, 'pressure'],
 'vibration': [1, 'vibration'],
 'errorID': [1, 'error'],
 'comp': [1, 'comp'],
 'failure': [1, 'failure'],
 'target': [1, 'target'],
 'volt_lag_1h': [2, 'volt'],
 'volt_lag_6h': [2, 'volt'],
 'volt_lag_12h': [2, 'volt'],
 'volt_lag_24h': [2, 'volt'],
 'volt_mean_24h': [2, 'volt'],
 'volt_std_24h': [2, 'volt'],
 'volt_min_24h': [2, 'volt'],
 'volt_max_24h': [2, 'volt'],
 'volt_mean_6h': [2, 'volt'],
 'volt_std_6h': [2, 'volt'],
 'rotate_lag_1h': [2, 'rotate'],
 'rotate_lag_6h': [2, 'rotate'],
 'rotate_lag_12h': [2, 'rotate'],
 'rotate_lag_24h': [2, 'rotate'],
 'rotate_mean_24h': [2, 'rotate'],
 'rotate_std_24h': [2, 'rotate'],
 'rotate_min_24h': [2, 'rotate'],
 'rotate_max_24h': [2, 'rotate'],
 'rotate_mean_6h': [2, 'rotate'],
 'rotate_std_6h': [2, 'rotate'],
 'pressure_lag_1h': [2, 'pressure'],
 'pressure_lag_6h': [2, 'pressure'],
 'pressure_lag_12h': [2, 'pressure'],
 'pressure_lag

In [10]:
print(format_taxonomy(gpt4o_taxonomy))

📌 **Taxonomy Structure:**

├── 📂 Time
  ├── 📄 datetime
  ├── 📄 hour
  ├── 📄 day_of_week
  ├── 📄 is_weekend
  ├── 📄 is_working_hours
  ├── 📄 hours_since_maint
  └── 📄 hours_since_error
├── 📂 Electrical
  └── 📂 volt
    ├── 📄 volt_lag_1h
    ├── 📄 volt_lag_6h
    ├── 📄 volt_lag_12h
    ├── 📄 volt_lag_24h
    ├── 📄 volt_mean_24h
    ├── 📄 volt_std_24h
    ├── 📄 volt_min_24h
    ├── 📄 volt_max_24h
    ├── 📄 volt_mean_6h
    └── 📄 volt_std_6h
├── 📂 Mechanical
  ├── 📂 rotate
    ├── 📄 rotate_lag_1h
    ├── 📄 rotate_lag_6h
    ├── 📄 rotate_lag_12h
    ├── 📄 rotate_lag_24h
    ├── 📄 rotate_mean_24h
    ├── 📄 rotate_std_24h
    ├── 📄 rotate_min_24h
    ├── 📄 rotate_max_24h
    ├── 📄 rotate_mean_6h
    └── 📄 rotate_std_6h
  ├── 📂 pressure
    ├── 📄 pressure_lag_1h
    ├── 📄 pressure_lag_6h
    ├── 📄 pressure_lag_12h
    ├── 📄 pressure_lag_24h
    ├── 📄 pressure_mean_24h
    ├── 📄 pressure_std_24h
    ├── 📄 pressure_min_24h
    ├── 📄 pressure_max_24h
    ├── 📄 pressure_mean_6h
    └── 📄 pressure_

In [11]:
print(format_taxonomy(gemma2_taxonomy))

📌 **Taxonomy Structure:**

├── 📄 Type
├── 📂 Time
  ├── 📄 datetime
  ├── 📄 hour
  ├── 📄 day_of_week
  ├── 📄 is_weekend
  └── 📄 is_working_hours
├── 📂 Sensor
  ├── 📂 volt
    ├── 📄 volt_lag_1h
    ├── 📄 volt_lag_6h
    ├── 📄 volt_lag_12h
    ├── 📄 volt_lag_24h
    ├── 📄 volt_mean_24h
    ├── 📄 volt_std_24h
    ├── 📄 volt_min_24h
    ├── 📄 volt_max_24h
    ├── 📄 volt_mean_6h
    └── 📄 volt_std_6h
  ├── 📂 rotate
    ├── 📄 rotate_lag_1h
    ├── 📄 rotate_lag_6h
    ├── 📄 rotate_lag_12h
    ├── 📄 rotate_lag_24h
    ├── 📄 rotate_mean_24h
    ├── 📄 rotate_std_24h
    ├── 📄 rotate_min_24h
    ├── 📄 rotate_max_24h
    ├── 📄 rotate_mean_6h
    └── 📄 rotate_std_6h
  ├── 📂 pressure
    ├── 📄 pressure_lag_1h
    ├── 📄 pressure_lag_6h
    ├── 📄 pressure_lag_12h
    ├── 📄 pressure_lag_24h
    ├── 📄 pressure_mean_24h
    ├── 📄 pressure_std_24h
    ├── 📄 pressure_min_24h
    ├── 📄 pressure_max_24h
    ├── 📄 pressure_mean_6h
    └── 📄 pressure_std_6h
  └── 📂 vibration
    ├── 📄 vibration_lag_1h
    ├── 📄 

In [12]:
print(format_taxonomy(llama3_taxonomy))

📌 **Taxonomy Structure:**

├── 📂 Time
  ├── 📄 datetime
  ├── 📄 hour
  ├── 📄 day_of_week
  ├── 📄 is_weekend
  └── 📄 is_working_hours
├── 📂 Sensor_Readings
  ├── 📂 volt
    ├── 📂 Volt_Lag
      ├── 📄 volt_lag_1h
      ├── 📄 volt_lag_6h
      ├── 📄 volt_lag_12h
      └── 📄 volt_lag_24h
    └── 📂 Volt_Aggregations
      ├── 📄 volt_mean_24h
      ├── 📄 volt_std_24h
      ├── 📄 volt_min_24h
      ├── 📄 volt_max_24h
      ├── 📄 volt_mean_6h
      └── 📄 volt_std_6h
  ├── 📂 rotate
    ├── 📂 Rotate_Lag
      ├── 📄 rotate_lag_1h
      ├── 📄 rotate_lag_6h
      ├── 📄 rotate_lag_12h
      └── 📄 rotate_lag_24h
    └── 📂 Rotate_Aggregations
      ├── 📄 rotate_mean_24h
      ├── 📄 rotate_std_24h
      ├── 📄 rotate_min_24h
      ├── 📄 rotate_max_24h
      ├── 📄 rotate_mean_6h
      └── 📄 rotate_std_6h
  ├── 📂 pressure
    ├── 📂 Pressure_Lag
      ├── 📄 pressure_lag_1h
      ├── 📄 pressure_lag_6h
      ├── 📄 pressure_lag_12h
      └── 📄 pressure_lag_24h
    └── 📂 Pressure_Aggregations
      ├── 📄 pressu

<br> <br> <br>

## Calculate the intra homogenity


In [13]:
def calculate_intra_group_homogeneity(taxonomy, df):
    """
    Calculate intra-group homogeneity scores based on correlation within taxonomy groups

    Parameters:
    taxonomy (dict): Dictionary with features as keys and [level, parent] as values
    df (pd.DataFrame): DataFrame containing the feature data

    Returns:
    dict: Dictionary with group names as keys and homogeneity scores as values
    """
    # Extract groups from taxonomy
    groups = {}
    for feature, (level, parent) in taxonomy.items():
        if feature in df.columns:  # Only consider features present in the dataframe
            if parent not in groups:
                groups[parent] = []
            groups[parent].append(feature)

    # Calculate homogeneity scores for each group
    homogeneity_scores = {}
    # Get only numeric columns from df
    numeric_df = df.select_dtypes(include=["number"])

    for parent, features in groups.items():
        # Filter to include only numeric features
        numeric_features = [f for f in features if f in numeric_df.columns]

        # Only calculate for groups with multiple numeric features
        if len(numeric_features) > 1:
            # Get the correlation matrix for features in this group
            corr_matrix = numeric_df[numeric_features].corr().abs()

            # Calculate the average correlation (excluding self-correlations)
            total_corr = 0
            count = 0
            for i in range(len(numeric_features)):
                for j in range(i + 1, len(numeric_features)):
                    total_corr += corr_matrix.iloc[i, j]
                    count += 1

            avg_corr = total_corr / count if count > 0 else 0
            homogeneity_scores[parent] = avg_corr

    return homogeneity_scores


In [15]:
openai_homogeneity = calculate_intra_group_homogeneity(gpt4o_taxonomy, df_machine_98)
llama33_homogeneity = calculate_intra_group_homogeneity(llama3_taxonomy, df_machine_98)
gemma2_homogeneity = calculate_intra_group_homogeneity(gemma2_taxonomy, df_machine_98)
deepseek_homogeneity = calculate_intra_group_homogeneity(deepseek_taxonomy, df_machine_98)

# Compare homogeneity scores
print("\n--- Intra-Group Homogeneity Comparison ---")

print("\nOpenAI Taxonomy Homogeneity Scores:")
for group, score in sorted(openai_homogeneity.items(), key=lambda x: x[1], reverse=True):
    print(f"{group}: {score:.4f}")

print("\nLlama 3.3 Taxonomy Homogeneity Scores:")
for group, score in sorted(llama33_homogeneity.items(), key=lambda x: x[1], reverse=True):
    print(f"{group}: {score:.4f}")

print("\nGemma2 Taxonomy Homogeneity Scores:")
for group, score in sorted(gemma2_homogeneity.items(), key=lambda x: x[1], reverse=True):
    print(f"{group}: {score:.4f}")

print("\nDeepSeek Taxonomy Homogeneity Scores:")
for group, score in sorted(deepseek_homogeneity.items(), key=lambda x: x[1], reverse=True):
    print(f"{group}: {score:.4f}")


# Calculate average homogeneity for each taxonomy
avg_openai = sum(openai_homogeneity.values()) / len(openai_homogeneity) if openai_homogeneity else 0
avg_llama33 = sum(llama33_homogeneity.values()) / len(llama33_homogeneity) if llama33_homogeneity else 0
avg_gemma2 = sum(gemma2_homogeneity.values()) / len(gemma2_homogeneity) if gemma2_homogeneity else 0
avg_deepseek = sum(deepseek_homogeneity.values()) / len(deepseek_homogeneity) if deepseek_homogeneity else 0

print(f"\nAverage Homogeneity - OpenAI: {round(avg_openai, 2)}, Llama 3.3: {round(avg_llama33, 2)}, Gemma2: {round(avg_gemma2, 2)}, DeepSeek: {round(avg_deepseek, 2)}")


--- Intra-Group Homogeneity Comparison ---

OpenAI Taxonomy Homogeneity Scores:
Maintenance: 0.3860
Error: 0.3738
pressure: 0.3029
vibration: 0.2484
rotate: 0.2227
volt: 0.2020
Outcome: 0.1269
Time: 0.0736
Mechanical: 0.0133
errorID: 0.0042
comp: 0.0035

Llama 3.3 Taxonomy Homogeneity Scores:
Maint_Count: 0.4982
Error_Count: 0.4978
Pressure_Aggregations: 0.3948
Vibration_Aggregations: 0.3722
Rotate_Aggregations: 0.3480
Volt_Aggregations: 0.3349
Pressure_Lag: 0.1898
Time: 0.1524
Root: 0.1269
Vibration_Lag: 0.1128
Rotate_Lag: 0.0999
Maintenance: 0.0912
Volt_Lag: 0.0633
Error: 0.0590
Sensor_Readings: 0.0118
errorID: 0.0042
comp: 0.0035

Gemma2 Taxonomy Homogeneity Scores:
pressure: 0.3029
vibration: 0.2484
rotate: 0.2227
volt: 0.2020
Status: 0.1593
Time: 0.1524
Sensor: 0.0118
errorID: 0.0042
comp: 0.0035

DeepSeek Taxonomy Homogeneity Scores:
pressure: 0.2967
vibration: 0.2400
rotate: 0.2132
volt: 0.1917
time: 0.1524
error: 0.1455
comp: 0.0545

Average Homogeneity - OpenAI: 0.18, Llama 3

In [17]:
def get_taxonomy_paths(taxonomy_json, node_names):
    """Returns hierarchical paths for given nodes in a taxonomy."""
    if isinstance(taxonomy_json, str):
        taxonomy_json = json.loads(taxonomy_json)  # Parse JSON if it's a string

    # Reverse lookup: Map each child to its parent
    child_to_parent = {child: parent for child, (_, parent) in taxonomy_json.items()}

    def get_path(node):
        """Recursively constructs the hierarchy path for a node."""
        path = [node]
        while node in child_to_parent and child_to_parent[node] != "Root":
            node = child_to_parent[node]
            path.append(node)
        return " --> ".join(reversed(path))  # Reverse to get correct hierarchy order
    return {node: get_path(node) for node in node_names}


In [19]:
df_machine_98.head()

,datetime,volt,rotate,pressure,vibration,errorID,comp,failure,target,volt_lag_1h,...,error_count_6h,error_count_24h,maint_count_6h,maint_count_24h,hour,day_of_week,is_weekend,is_working_hours,hours_since_maint,hours_since_error
0,2015-01-01 06:00:00,0.303137,0.604476,0.271993,0.621000,0,0,0,0,0.000000,...,0.0,0.0,0.0,0.0,6,3,0,0,0,0
1,2015-01-01 07:00:00,0.455312,0.635538,0.480756,0.645421,0,0,0,0,0.303137,...,0.0,0.0,0.0,0.0,7,3,0,0,1,1
2,2015-01-01 08:00:00,0.449470,0.540973,0.427196,0.356126,0,0,0,0,0.455312,...,0.0,0.0,0.0,0.0,8,3,0,1,2,2
3,2015-01-01 09:00:00,0.408248,0.687752,0.378658,0.254123,0,0,0,0,0.449470,...,0.0,0.0,0.0,0.0,9,3,0,1,3,3
4,2015-01-01 10:00:00,0.486041,0.518746,0.454206,0.351050,0,0,0,0,0.408248,...,0.0,0.0,0.0,0.0,10,3,0,1,4,4


In [31]:
# Test the function with the openai_taxonomy data

shap_features = {
    'feature': ["volt", 'rotate', 'vibration',  'vibration_lag_12h']
}


# features_to_find = shap_features["feature"].tolist()
features_to_find = shap_features["feature"]

taxonomy_model = [gpt4o_taxonomy, llama3_taxonomy, gemma2_taxonomy, deepseek_taxonomy]
index = 1

match index:
    case 0:
        print("GPT-4o")
    case 1:
        print("Llama 3.3")
    case 2:
        print("Gemma2")
    case 3:
        print("DeepSeek")
    case _:
        print("Unknown model")

try:
    # Get paths for all selected features
    feature_paths = get_taxonomy_paths(taxonomy_model[index], features_to_find)

    # Print the results in a formatted way
    print("\n=== Taxonomy Paths ===\n")
    for feature, path in feature_paths.items():
        print(f"{feature}: {path}")
except IndexError as e:
    print("Taxonomy model index is out of reach")

Llama 3.3

=== Taxonomy Paths ===

volt: Sensor_Readings --> volt
rotate: Sensor_Readings --> rotate
vibration: Sensor_Readings --> vibration
vibration_lag_12h: Sensor_Readings --> vibration --> Vibration_Lag --> vibration_lag_12h
